# OASIS INFOBYTE — Data Analytics Task 2
## Customer Segmentation Analysis

**Objective:** Segment an e-commerce customer base into distinct groups based on purchasing behaviour using RFM features and K-Means clustering.

**Dataset:** UCI Online Retail dataset
**Tools:** Python, pandas, scikit-learn (KMeans), matplotlib, seaborn, Jupyter Notebook


### Project Checklist
- Load and inspect the dataset
- Handle missing and inconsistent data
- Calculate average purchase value, purchase frequency and customer lifetime value
- Build RFM features: Recency, Frequency, Monetary
- Standardise features with `StandardScaler`
- Use the Elbow Method to select K
- Apply K-Means clustering
- Visualise clusters with at least two feature combinations
- Profile each cluster
- Show customer count per cluster
- Give marketing recommendations for every segment


In [ ]:
!pip -q install ucimlrepo openpyxl

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')


## 1. Load the UCI Online Retail Dataset
The UCI Online Retail dataset contains transaction-level records from a UK-based non-store online retailer. The dataset is suitable for clustering/customer segmentation.

In [ ]:
from ucimlrepo import fetch_ucirepo

online_retail = fetch_ucirepo(id=352)
df = online_retail.data.features.copy()

print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
print('Column names:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))


## 2. Data Cleaning
For customer-level segmentation, rows without `CustomerID` cannot be assigned to a customer and are removed. Cancelled invoices are excluded using the `C` prefix in `InvoiceNo`. Non-positive quantities/prices are also removed so that revenue and purchasing behaviour are not distorted.

In [ ]:
df = df.copy()

# Standardise column names in case the source uses spaces/case differences
df.columns = [c.strip() for c in df.columns]

# Convert date column
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Record cleaning counts
before = len(df)

df = df.dropna(subset=['CustomerID', 'InvoiceDate'])
after_customer = len(df)

df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df = df[~df['InvoiceNo'].str.upper().str.startswith('C')]
after_cancel = len(df)

df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
after_positive = len(df)

df = df.drop_duplicates()
after_duplicates = len(df)

df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

print(f'Rows before cleaning: {before:,}')
print(f'After removing missing CustomerID/date: {after_customer:,}')
print(f'After removing cancelled invoices: {after_cancel:,}')
print(f'After removing non-positive quantity/price: {after_positive:,}')
print(f'After removing exact duplicates: {after_duplicates:,}')
display(df.head())


## 3. Descriptive Customer Statistics
We calculate:
- **Average Purchase Value:** total monetary value divided by number of invoices.
- **Purchase Frequency:** number of unique invoices.
- **Customer Lifetime Value (observed period):** total monetary value generated by the customer during the dataset period.

In [ ]:
customer_stats = df.groupby('CustomerID').agg(
    total_spend=('TotalAmount', 'sum'),
    purchase_frequency=('InvoiceNo', 'nunique'),
    first_purchase=('InvoiceDate', 'min'),
    last_purchase=('InvoiceDate', 'max')
).reset_index()

customer_stats['average_purchase_value'] = (
    customer_stats['total_spend'] / customer_stats['purchase_frequency']
)
customer_stats['customer_lifetime_value'] = customer_stats['total_spend']

display(customer_stats[['average_purchase_value','purchase_frequency','customer_lifetime_value']].describe().T)


## 4. RFM Analysis

**Recency:** days since the customer's most recent purchase.

**Frequency:** number of unique invoices/orders.

**Monetary:** total customer spend.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalAmount', 'sum')
).reset_index()

display(rfm.head())
display(rfm[['Recency','Frequency','Monetary']].describe().T)


## 5. RFM Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(rfm['Recency'], bins=40, kde=True, ax=axes[0])
axes[0].set_title('Recency Distribution')
sns.histplot(rfm['Frequency'], bins=40, kde=True, ax=axes[1])
axes[1].set_title('Frequency Distribution')
sns.histplot(rfm['Monetary'], bins=40, kde=True, ax=axes[2])
axes[2].set_title('Monetary Distribution')
plt.tight_layout()
plt.show()


## 6. Standardise RFM Features
K-Means is distance-based, so the RFM variables are standardised before clustering.

In [ ]:
features = ['Recency', 'Frequency', 'Monetary']
X = rfm[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=rfm.index)
display(X_scaled_df.head())


## 7. Elbow Method — Find the Number of Clusters

In [ ]:
inertias = []
K_range = range(2, 9)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(list(K_range), inertias, marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.xticks(list(K_range))
plt.show()

elbow_table = pd.DataFrame({'K': list(K_range), 'Inertia': inertias})
display(elbow_table)


### Cluster selection
After inspecting the elbow curve above, the notebook uses **K = 4** as the working segmentation choice. If your generated elbow plot shows a clearer elbow at another value, change `OPTIMAL_K` below and rerun the clustering cells.

In [ ]:
OPTIMAL_K = 4

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(X_scaled) + 1

print('Selected K:', OPTIMAL_K)
print('Silhouette score:', round(silhouette_score(X_scaled, kmeans.labels_), 3))
display(rfm.head())


## 8. Visualise Customer Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster', palette='tab10', ax=axes[0], alpha=0.7)
axes[0].set_title('Clusters: Recency vs Monetary')
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Cluster', palette='tab10', ax=axes[1], alpha=0.7)
axes[1].set_title('Clusters: Frequency vs Monetary')
plt.tight_layout()
plt.show()


## 9. Cluster Profiling

In [ ]:
cluster_profile = rfm.groupby('Cluster').agg(
    Customers=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).round(2).sort_values('Avg_Monetary', ascending=False)

cluster_profile['Revenue_Share_%'] = (
    cluster_profile['Total_Revenue'] / cluster_profile['Total_Revenue'].sum() * 100
).round(2)

display(cluster_profile)


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x=cluster_profile.index.astype(str), y=cluster_profile['Customers'].values)
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.title('Number of Customers per Cluster')
plt.show()


## 10. Customer Segment Interpretation
Cluster labels are assigned from the calculated profile rather than assuming that a cluster number has a fixed meaning.

- **Champions / High-value:** low recency, high frequency and high monetary value → reward and retain.
- **Loyal / Regular:** relatively frequent purchasers with solid spend → loyalty offers and cross-selling.
- **Potential / New:** relatively recent customers but lower frequency/spend → onboarding and conversion campaigns.
- **At Risk / Hibernating:** high recency (long time since purchase) with previous activity → win-back campaigns and personalised offers.

Use the numerical `cluster_profile` table above to confirm the exact interpretation for this run.

## 11. Marketing Recommendations
| Segment profile | Recommended action |
|---|---|
| High-value / Champions | VIP rewards, early access, premium products, referral incentives |
| Loyal / Regular | Loyalty points, bundles, cross-selling and personalised recommendations |
| New / Potential | Welcome offers, second-purchase discounts, product education |
| At Risk / Hibernating | Win-back email/SMS, limited-time offers, personalised reactivation campaigns |

These recommendations should be mapped to the actual cluster characteristics shown in the profiling table.

## 12. Final Conclusion
K-Means clustering was applied to standardised RFM features to divide customers into behaviour-based groups. The resulting cluster profiles help the business identify high-value customers, regular customers, newer/potential customers and customers who may need reactivation. This segmentation can support targeted marketing instead of using the same campaign for every customer.

### Key business questions answered
1. Who purchases recently?
2. Who purchases frequently?
3. Who generates the most monetary value?
4. How many customers belong to each segment?
5. What marketing action should be used for each segment?

## Dataset Citation
Chen, D. (2015). **Online Retail [Dataset]. UCI Machine Learning Repository.** DOI: 10.24432/C5BW33.

This notebook follows the Oasis Infobyte Task 2 requirements: RFM analysis, standardisation, K-Means, Elbow Method, cluster visualisation, cluster profiling, customer-count chart and marketing recommendations.